# Nhóm 21 - Apriori bằng Julia

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/anhJ04/Frequent-Itemset.git"
PROJECT_SUBDIR = "21"                                                  

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    %cd /content/repo/{PROJECT_SUBDIR}
    if subprocess.run("which julia", shell=True, capture_output=True).returncode != 0:
        !curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o /tmp/julia.tar.gz
        !tar -xzf /tmp/julia.tar.gz -C /usr --strip-components 1
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Thư mục làm việc:", os.getcwd())
!julia --version
!julia --project=. -e 'using Pkg; Pkg.instantiate()'

## 1. Unit test

So sánh 4 biến thể với kết quả của SPMF trên 5 CSDL và với brute force trên dữ liệu ngẫu nhiên.

In [ ]:
!julia --project=. tests/runtests.jl

## 2. Chạy CLI và so sánh với SPMF

In [ ]:
!julia --project=. run_cli.jl --input data/benchmark/chess.txt --minsup 0.8 --algo hybrid -o results/demo/chess_0.8
!python compare_outputs.py tests/ref/chess_0.8.txt results/demo/chess_0.8_list.txt
!head -5 results/demo/chess_0.8_list.txt

## 3. Thực nghiệm

Ô dưới chạy bản rút gọn (Chess, Mushroom) để kiểm tra trên Colab. Kết quả đầy đủ chạy trên máy nhóm, lưu trong `results/exp/`.

In [ ]:
!rm -rf results/exp_colab
!python bench/run_experiments.py --datasets chess mushrooms --exps runtime --algos optimized hybrid --no-spmf --cap 30 --out results/exp_colab
!python bench/plot_results.py --dir results/exp_colab
import pandas as pd
pd.read_csv("results/exp_colab/runtime.csv")[["dataset", "algo", "minsup", "count", "max_k", "time", "status"]]

In [ ]:
from IPython.display import Image, display
for d in ["results/exp_colab/fig", "results/exp/fig"]:
    for name in ["runtime_vs_minsup", "count_vs_minsup", "memory_peak", "scalability", "length_effect"]:
        p = f"{d}/{name}.png"
        if os.path.exists(p):
            print(p); display(Image(p, width=650))

## 4. Ứng dụng 1: Phân tích giỏ hàng (Groceries)

Dữ liệu: 9.835 hoá đơn, 169 nhóm hàng.

Cách biến đổi: mỗi hoá đơn là một giao dịch, mỗi nhóm hàng được đánh số thành một item. Dữ liệu thưa (khoảng 4,4 món mỗi hoá đơn) nên hợp với Apriori.

Ngưỡng: minsup = 1%, minconf = 30%.

In [ ]:
import csv, json
from collections import Counter
rows = []
with open("data/raw/groceries.csv", encoding="utf-8") as f:
    for r in csv.reader(f):
        items = sorted({x.strip() for x in r if x.strip()})
        if items:
            rows.append(items)
names = sorted({x for r in rows for x in r})
name2id = {n: i for i, n in enumerate(names, 1)}
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/groceries.txt", "w") as f:
    for r in rows:
        f.write(" ".join(str(name2id[x]) for x in sorted(r, key=name2id.get)) + "\n")
json.dump({str(i): n for n, i in name2id.items()}, open("data/processed/groceries_items.json", "w"),
          ensure_ascii=False, indent=0)
N = len(rows)
print(f"{N} giao dịch | {len(names)} item | độ dài TB {sum(map(len, rows)) / N:.2f}")
print("Mặt hàng phổ biến nhất:", Counter(x for r in rows for x in r).most_common(8))

In [ ]:
!julia --project=. apps/app_rules.jl data/processed/groceries.txt 0.01 0.3 results/app1/groceries_rules.csv hybrid

In [ ]:
id2name = json.load(open("data/processed/groceries_items.json"))
dec = lambda s: ", ".join(id2name[i] for i in str(s).split())
rules = pd.read_csv("results/app1/groceries_rules.csv")
rules["Nếu mua (X)"] = rules["antecedent"].map(dec)
rules["thì mua (Y)"] = rules["consequent"].map(dec)
print(f"Tổng số luật (support ≥ 1%, confidence ≥ 30%): {len(rules)}")
top10 = rules.head(10)[["Nếu mua (X)", "thì mua (Y)", "support", "confidence", "lift"]].round(3)
top10.index = range(1, 11)
top10

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.6))
sc = ax.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="Blues", s=22,
                edgecolors="#52514d", linewidths=0.3)
ax.set_xlabel("support"); ax.set_ylabel("confidence")
ax.set_title("Luật kết hợp trên Groceries (màu = lift)")
fig.colorbar(sc, label="lift"); ax.grid(color="#e4e3df", linewidth=0.6)
plt.tight_layout(); plt.savefig("results/app1/rules_scatter.png", dpi=200); plt.show()

In [ ]:
# Độ nhạy theo ngưỡng: chạy lại cài đặt Julia của nhóm với nhiều cặp (minsup, minconf)
rows_s = []
for ms in (0.02, 0.01, 0.005):
    for mc in (0.3, 0.5):
        out = f"results/app1/sens_{ms}_{mc}.csv"
        subprocess.run(["julia", "--project=.", "apps/app_rules.jl", "data/processed/groceries.txt",
                        str(ms), str(mc), out, "hybrid"], check=True, capture_output=True)
        r = pd.read_csv(out)
        n_fi = sum(1 for l in open(out.replace(".csv", "_itemsets.txt")) if "#SUP" in l)
        rows_s.append({"minsup": ms, "minconf": mc, "#tập phổ biến": n_fi, "#luật": len(r),
                       "lift lớn nhất": round(r["lift"].max(), 2) if len(r) else None})
print("Vế phải phổ biến nhất của các luật (minsup 1%, minconf 30%):",
      rules["thì mua (Y)"].value_counts().head(5).to_dict())
print("Số luật theo |X|:", rules["antecedent"].astype(str).str.split().str.len().value_counts().sort_index().to_dict())
pd.DataFrame(rows_s)

### Nhận xét

- Phần lớn luật top đầu là rau củ và trái cây đi cùng nhau, lift khoảng 3. Khách mua hai loại đồ tươi thì hay mua thêm loại thứ ba. Nên xếp các quầy này gần nhau hoặc làm combo.
- beef ⇒ root vegetables (lift 3,04): thịt bò thường mua kèm rau củ để nấu. Có thể bày rau củ gần quầy thịt.
- Nhóm sữa: curd, whole milk ⇒ yogurt. Có thể làm gói bữa sáng sữa chua và trái cây.
- whole milk và other vegetables xuất hiện trong rất nhiều luật vì bản thân chúng được mua nhiều. Vì vậy nhóm xếp theo lift thay vì confidence.
- Hạn chế: lift chỉ cho biết hai món hay đi cùng nhau, không phải nguyên nhân. Dữ liệu chỉ có 30 ngày của một cửa hàng.

## 5. Ứng dụng 2: Genome SARS-CoV-2

Dữ liệu: genome MT745584, dài 29.860 nucleotide.

Cách biến đổi: cắt genome thành các đoạn 100 nucleotide, bước 50. Mỗi đoạn là một giao dịch, item là các 3-mer (chuỗi 3 nucleotide) có trong đoạn.

Nhóm có thử k = 6 nhưng dữ liệu quá thưa, chỉ ra itemset 1 phần tử. Với k = 3 mới có các tổ hợp để phân tích (bảng ở ô dưới).

In [ ]:
# Chọn tham số biến đổi: thử nhiều cấu hình (k, W, S, minsup) bằng cài đặt Julia của nhóm
from itertools import product
genome = "".join(l.strip().upper() for l in open("data/benchmark/MT745584.txt")
                 if l.strip() and not l.startswith(">"))
def make_tx(K, W, S, path):
    km = {"".join(p): i for i, p in enumerate(product("ACGT", repeat=K), 1)}
    with open(path, "w") as f:
        n = 0
        for s in range(0, len(genome) - W + 1, S):
            w = genome[s:s + W]
            ids = sorted({km[w[j:j + K]] for j in range(W - K + 1) if w[j:j + K] in km})
            f.write(" ".join(map(str, ids)) + "\n"); n += 1
    return n
os.makedirs("results/app2", exist_ok=True)
grid_rows = []
for K, W, S, ms in [(6, 100, 95, 0.1), (4, 100, 50, 0.5), (4, 150, 75, 0.6), (3, 100, 50, 0.8)]:
    path = f"data/processed/covid_k{K}_w{W}_s{S}.txt"
    n = make_tx(K, W, S, path)
    out = f"results/app2/grid_k{K}_w{W}_s{S}_{ms}"
    subprocess.run(["julia", "--project=.", "run_cli.jl", "--input", path, "--minsup", str(ms),
                    "--algo", "hybrid", "-o", out], check=True, capture_output=True)
    sizes = Counter(len(l.split("#SUP:")[0].split()) for l in open(out + "_list.txt") if "#SUP" in l)
    grid_rows.append({"k": K, "W": W, "S": S, "minsup": ms, "#giao dịch": n, "#itemset": sum(sizes.values()),
                      "theo kích thước": dict(sorted(sizes.items()))})
pd.DataFrame(grid_rows)

In [ ]:
from itertools import product
K, W, S = 3, 100, 50
genome = "".join(l.strip().upper() for l in open("data/benchmark/MT745584.txt")
                 if l.strip() and not l.startswith(">"))
kmer2id = {"".join(p): i for i, p in enumerate(product("ACGT", repeat=K), 1)}
id2kmer = {str(i): k for k, i in kmer2id.items()}
starts = list(range(0, len(genome) - W + 1, S))
with open("data/processed/covid_k3.txt", "w") as f:
    for s in starts:
        w = genome[s:s + W]
        ids = sorted({kmer2id[w[j:j + K]] for j in range(W - K + 1) if w[j:j + K] in kmer2id})
        f.write(" ".join(map(str, ids)) + "\n")
gc = (genome.count("G") + genome.count("C")) / len(genome)
print(f"Genome: {len(genome):,} nt | GC = {gc:.1%} | {len(starts)} giao dịch | {len(kmer2id)} item (3-mer)")

In [ ]:
# (a) tổ hợp 3-mer phổ biến: minsup 80% và 70% — chạy bằng cài đặt Julia của nhóm
!julia --project=. run_cli.jl --input data/processed/covid_k3.txt --minsup 0.8 --algo hybrid -o results/app2/covid_k3_0.8
!julia --project=. run_cli.jl --input data/processed/covid_k3.txt --minsup 0.7 --algo hybrid -o results/app2/covid_k3_0.7
# (b) support của cả 64 3-mer (chỉ mức 1: --maxlen 1, minsup rất thấp)
!julia --project=. run_cli.jl --input data/processed/covid_k3.txt --minsup 0.001 --maxlen 1 --algo hybrid -o results/app2/covid_k3_single

In [ ]:
def load_fi(path):
    out = []
    for line in open(path):
        if "#SUP:" in line:
            left, sup = line.split("#SUP:")
            out.append(([id2kmer[i] for i in left.split()], int(sup)))
    return out

Nt = len(starts)
for ms in ("0.8", "0.7"):
    fi = load_fi(f"results/app2/covid_k3_{ms}_list.txt")
    by = Counter(len(k) for k, _ in fi)
    n_cg = sum(any("CG" in x for x in k) for k, _ in fi)
    print(f"minsup={ms}: {len(fi)} itemset, theo kích thước {dict(sorted(by.items()))}, "
          f"số itemset chứa 3-mer có 'CG': {n_cg}")

fi = load_fi("results/app2/covid_k3_0.8_list.txt")
top3 = sorted((x for x in fi if len(x[0]) == 3), key=lambda x: -x[1])[:10]
pd.DataFrame([{"tổ hợp 3-mer": " + ".join(k), "support": s, "% đoạn genome": round(100 * s / Nt, 1)}
              for k, s in top3], index=range(1, len(top3) + 1))

In [ ]:
single = sorted(((k[0], s / Nt) for k, s in load_fi("results/app2/covid_k3_single_list.txt")),
                key=lambda x: x[1])
has_cg = [("CG" in k) for k, _ in single]
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(single)), [s for _, s in single], width=0.9,
       color=["#eb6834" if c else "#2a78d6" for c in has_cg])
ax.set_xticks(range(len(single))); ax.set_xticklabels([k for k, _ in single], rotation=90, fontsize=7)
ax.axhline(0.8, color="#52514d", lw=0.8, ls="--"); ax.text(0, 0.82, "minsup = 0.8", fontsize=8)
ax.set_ylabel("support (tỉ lệ đoạn)")
ax.set_title("Support của 64 3-mer (cam: chứa dinucleotide CG, xanh: không chứa)")
ax.grid(axis="y", color="#e4e3df", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig("results/app2/kmer_support.png", dpi=200); plt.show()

print("8 3-mer hiếm nhất:", [(k, round(s, 3)) for k, s in single[:8]])
# tỉ lệ quan sát / kỳ vọng của dinucleotide CG (đo mức ức chế CpG)
L = len(genome)
obs = sum(genome[i:i + 2] == "CG" for i in range(L - 1)) / (L - 1)
exp = genome.count("C") / L * genome.count("G") / L
print(f"CpG quan sát/kỳ vọng = {obs / exp:.2f}  (1.0 = không thiên lệch)")

### Nhận xét

- Không có itemset phổ biến nào chứa 3-mer có "CG". Cả 8 3-mer chứa CG đều nằm trong nhóm hiếm nhất. Tỉ lệ CG quan sát/kỳ vọng khoảng 0,4. Đây là hiện tượng ức chế CpG đã được ghi nhận ở SARS-CoV-2, thường được giải thích là virus tránh bị protein ZAP của vật chủ nhận diện.
- Các tổ hợp phổ biến nhất giàu T (U trên RNA), đúng với việc genome này có GC thấp (38%).
- Có thể dùng các tập k-mer phổ biến làm đặc trưng để phân loại virus hoặc tìm đoạn genome bất thường.
- Hạn chế: chỉ dùng một genome, nên kết quả phản ánh thành phần nucleotide chứ chưa phải motif chức năng.